# Prepare Transit Catchment Diversity Data (Sweden)

This notebook prepares all necessary data for computing transit catchment diversity in Sweden,
**processing each county separately** for computational feasibility.

**Per-county processing**:
1. **Origins (POIs)**: Exact county boundary - only POIs within the county
2. **Destinations (DeSO)**: Buffered boundary - DeSO centroids within county + 20km buffer

The 20km buffer ensures that POIs near county borders can reach DeSO zones just outside,
accounting for cross-border transit accessibility.

**Coverage**: 15 southern Swedish counties:
- Stockholm (01), Uppsala (03), Södermanland (04), Östergötland (05)
- Jönköping (06), Kronoberg (07), Kalmar (08), Gotland (09)
- Blekinge (10), Skåne (12), Halland (13), Västra Götaland (14)
- Värmland (17), Örebro (18), Västmanland (19)

**GTFS Structure**: `dbs/gtfs/sweden_south/c_XX/` where XX is county code

In [12]:
%cd /workspace

from __future__ import annotations

import json
from pathlib import Path

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import zipfile

# Directories
DESO_DIR = Path('dbs/deso')
FOOT_TRAFFIC_DIR = Path('dbs/sweden_weekly_patterns')
GTFS_BASE_DIR = Path('dbs/gtfs/sweden_south')
ROUTING_DIR = Path('dbs/routing')

# Buffer distance in meters for destination filtering
BUFFER_METERS = 20000  # 20 km

print(f"DeSO data: {DESO_DIR}")
print(f"Foot traffic data: {FOOT_TRAFFIC_DIR}")
print(f"GTFS base directory: {GTFS_BASE_DIR}")
print(f"Buffer distance: {BUFFER_METERS / 1000:.0f} km")

/workspace
DeSO data: dbs/deso
Foot traffic data: dbs/sweden_weekly_patterns
GTFS base directory: dbs/gtfs/sweden_south
Buffer distance: 20 km


In [2]:
# Swedish county codes for the 15 southern counties
SOUTH_COUNTY_CODES = [
    '01',  # Stockholm
    '03',  # Uppsala
    '04',  # Södermanland
    '05',  # Östergötland
    '06',  # Jönköping
    '07',  # Kronoberg
    '08',  # Kalmar
    '09',  # Gotland
    '10',  # Blekinge
    '12',  # Skåne
    '13',  # Halland
    '14',  # Västra Götaland
    '17',  # Värmland
    '18',  # Örebro
    '19',  # Västmanland
]

COUNTY_NAMES = {
    '01': 'Stockholm',
    '03': 'Uppsala',
    '04': 'Södermanland',
    '05': 'Östergötland',
    '06': 'Jönköping',
    '07': 'Kronoberg',
    '08': 'Kalmar',
    '09': 'Gotland',
    '10': 'Blekinge',
    '12': 'Skåne',
    '13': 'Halland',
    '14': 'Västra Götaland',
    '17': 'Värmland',
    '18': 'Örebro',
    '19': 'Västmanland',
}

# Map county code to SafeGraph REGION names (with encoding variants)
COUNTY_TO_REGION = {
    '01': ['Stockholm County'],
    '03': ['Uppsala County'],
    '04': ['Södermanland County', 'S\xf6dermanland County'],
    '05': ['Östergötland County', '\xd6sterg\xf6tland County'],
    '06': ['Jönköping County', 'J\xf6nk\xf6ping County'],
    '07': ['Kronoberg County'],
    '08': ['Kalmar County'],
    '09': ['Gotland County'],
    '10': ['Blekinge County'],
    '12': ['Skåne County', 'Sk\xe5ne County'],
    '13': ['Halland County'],
    '14': ['Västra Götaland County', 'V\xe4stra G\xf6taland County'],
    '17': ['Värmland County', 'V\xe4rmland County'],
    '18': ['Örebro County', '\xd6rebro County'],
    '19': ['Västmanland County', 'V\xe4stmanland County'],
}

print(f"Target counties: {len(SOUTH_COUNTY_CODES)}")

Target counties: 15


## 1. Load Base Data

Load DeSO zones and foot traffic data once, then filter per county.

In [3]:
# Load DeSO GeoPackage with geometry
deso_gdf = gpd.read_file(DESO_DIR / 'deso_harmonized_2024.gpkg')
deso_gdf['county_code'] = deso_gdf['deso_code'].str[:2]

print(f"Total DeSO zones: {len(deso_gdf)}")
print(f"CRS: {deso_gdf.crs}")

# Filter to southern counties
south_deso = deso_gdf[deso_gdf['county_code'].isin(SOUTH_COUNTY_CODES)].copy()
print(f"Southern Sweden DeSO zones: {len(south_deso)}")

ERROR 1: PROJ: proj_create_from_database: Open of /opt/conda/envs/geoenv/share/proj failed


Total DeSO zones: 6160
CRS: EPSG:3006
Southern Sweden DeSO zones: 5237


In [4]:
# Load aggregated Swedish foot traffic data
foot_traffic_file = FOOT_TRAFFIC_DIR / 'sweden_weekly_patterns_2024.parquet'
print(f"Loading: {foot_traffic_file}")

df_traffic = pd.read_parquet(
    foot_traffic_file,
    columns=['PLACEKEY', 'LATITUDE', 'LONGITUDE', 'REGION', 'TOP_CATEGORY']
)

# Get unique POIs
unique_pois = df_traffic.drop_duplicates(subset='PLACEKEY').copy()
print(f"Total unique POIs: {len(unique_pois):,}")

# Filter to southern counties (all REGION names)
all_south_regions = [r for regions in COUNTY_TO_REGION.values() for r in regions]
south_pois = unique_pois[unique_pois['REGION'].isin(all_south_regions)].copy()
print(f"Southern Sweden POIs: {len(south_pois):,}")

Loading: dbs/sweden_weekly_patterns/sweden_weekly_patterns_2024.parquet
Total unique POIs: 514,764
Southern Sweden POIs: 435,278


## 2. Per-County Data Preparation

For each county:
1. **Origins (POIs)**: Filter by exact county (REGION column)
2. **Destinations (DeSO)**: 
   - Create convex hull of county DeSO zones
   - Apply 20km buffer
   - Select all DeSO zones (from any county) that intersect the buffered area
   - Compute centroids in WGS84

In [5]:
def prepare_county_data(county_code: str, deso_gdf: gpd.GeoDataFrame, pois_df: pd.DataFrame) -> dict:
    """
    Prepare origins (POIs) and destinations (DeSO centroids) for a single county.
    
    Origins: Exact county boundary (POIs in this county only)
    Destinations: County convex hull + 20km buffer (DeSO zones that intersect)
    
    Returns dict with 'origins', 'destinations', 'stats'
    """
    county_name = COUNTY_NAMES[county_code]
    region_names = COUNTY_TO_REGION[county_code]
    
    # === ORIGINS: Exact county POIs ===
    county_pois = pois_df[pois_df['REGION'].isin(region_names)].copy()
    
    origins = county_pois[['PLACEKEY', 'LATITUDE', 'LONGITUDE']].rename(columns={
        'PLACEKEY': 'id',
        'LATITUDE': 'lat',
        'LONGITUDE': 'lon'
    }).dropna()
    
    # === DESTINATIONS: Buffered county DeSO zones ===
    # Get DeSO zones in this county
    county_deso = deso_gdf[deso_gdf['county_code'] == county_code].copy()
    
    if len(county_deso) == 0:
        return None
    
    # Create convex hull of county and buffer by 20km
    # Note: deso_gdf is in EPSG:3006 (SWEREF99 TM, meters)
    county_hull = county_deso.unary_union.convex_hull
    buffered_hull = county_hull.buffer(BUFFER_METERS)
    
    # Create GeoDataFrame for spatial join
    buffer_gdf = gpd.GeoDataFrame(geometry=[buffered_hull], crs=deso_gdf.crs)
    
    # Find all DeSO zones (from any county) that intersect the buffer
    # Use centroids for the intersection test
    all_deso = deso_gdf.copy()
    all_deso['centroid_geom'] = all_deso.geometry.centroid
    
    # Check which centroids are within the buffer
    centroids_gdf = gpd.GeoDataFrame(
        all_deso[['deso_code', 'county_code']], 
        geometry=all_deso['centroid_geom'],
        crs=deso_gdf.crs
    )
    
    # Spatial join to find DeSO zones within buffer
    deso_in_buffer = gpd.sjoin(centroids_gdf, buffer_gdf, predicate='within')
    deso_codes_in_buffer = deso_in_buffer['deso_code'].unique()
    
    # Get full DeSO data for zones in buffer
    buffered_deso = deso_gdf[deso_gdf['deso_code'].isin(deso_codes_in_buffer)].copy()
    
    # Compute centroids and convert to WGS84
    buffered_deso['centroid'] = buffered_deso.geometry.centroid
    centroids_wgs84 = gpd.GeoDataFrame(
        geometry=buffered_deso['centroid'], 
        crs='EPSG:3006'
    ).to_crs('EPSG:4326')
    
    buffered_deso['lon'] = centroids_wgs84.geometry.x
    buffered_deso['lat'] = centroids_wgs84.geometry.y
    buffered_deso['county'] = buffered_deso['county_code'].map(COUNTY_NAMES)
    
    destinations = buffered_deso[['deso_code', 'lat', 'lon', 'county', 'county_code']].rename(
        columns={'deso_code': 'id'}
    )
    
    # Stats
    n_core_deso = len(county_deso)
    n_buffer_deso = len(buffered_deso)
    n_extra = n_buffer_deso - n_core_deso
    
    stats = {
        'county_code': county_code,
        'county_name': county_name,
        'n_origins': len(origins),
        'n_destinations_core': n_core_deso,
        'n_destinations_buffer': n_buffer_deso,
        'n_destinations_extra': n_extra,
    }
    
    return {
        'origins': origins,
        'destinations': destinations,
        'stats': stats
    }

In [6]:
# Process all counties
all_stats = []

for county_code in SOUTH_COUNTY_CODES:
    print(f"\nProcessing county {county_code} ({COUNTY_NAMES[county_code]})...")
    
    result = prepare_county_data(county_code, south_deso, south_pois)
    
    if result is None:
        print(f"  Skipped - no DeSO zones found")
        continue
    
    # Create output directory
    output_dir = ROUTING_DIR / f'sweden_c_{county_code}'
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Save origins
    origins_path = output_dir / 'poi_locations.csv'
    result['origins'].to_csv(origins_path, index=False)
    
    # Save destinations
    destinations_path = output_dir / 'deso_centroids.csv'
    result['destinations'].to_csv(destinations_path, index=False)
    
    stats = result['stats']
    all_stats.append(stats)
    
    print(f"  Origins (POIs): {stats['n_origins']:,}")
    print(f"  Destinations (DeSO): {stats['n_destinations_buffer']:,} ({stats['n_destinations_core']:,} core + {stats['n_destinations_extra']:,} buffer)")
    print(f"  Saved to: {output_dir}")


Processing county 01 (Stockholm)...


/tmp/ipykernel_90297/1271544895.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  county_hull = county_deso.unary_union.convex_hull


  Origins (POIs): 124,106
  Destinations (DeSO): 1,587 (1,362 core + 225 buffer)
  Saved to: dbs/routing/sweden_c_01

Processing county 03 (Uppsala)...


/tmp/ipykernel_90297/1271544895.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  county_hull = county_deso.unary_union.convex_hull


  Origins (POIs): 18,124
  Destinations (DeSO): 631 (249 core + 382 buffer)
  Saved to: dbs/routing/sweden_c_03

Processing county 04 (Södermanland)...


...

  Origins (POIs): 13,249
  Destinations (DeSO): 247 (181 core + 66 buffer)
  Saved to: dbs/routing/sweden_c_17

Processing county 18 (Örebro)...


/tmp/ipykernel_90297/1271544895.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  county_hull = county_deso.unary_union.convex_hull


  Origins (POIs): 13,469
  Destinations (DeSO): 283 (184 core + 99 buffer)
  Saved to: dbs/routing/sweden_c_18

Processing county 19 (Västmanland)...


/tmp/ipykernel_90297/1271544895.py:31: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  county_hull = county_deso.unary_union.convex_hull


  Origins (POIs): 11,919
  Destinations (DeSO): 300 (159 core + 141 buffer)
  Saved to: dbs/routing/sweden_c_19


In [7]:
# Summary table
stats_df = pd.DataFrame(all_stats)
stats_df = stats_df.sort_values('n_origins', ascending=False)

print("\n" + "=" * 80)
print("PER-COUNTY DATA SUMMARY")
print("=" * 80)
print(stats_df.to_string(index=False))

print(f"\nTotal origins: {stats_df['n_origins'].sum():,}")
print(f"Total core destinations: {stats_df['n_destinations_core'].sum():,}")


PER-COUNTY DATA SUMMARY
county_code     county_name  n_origins  n_destinations_core  n_destinations_buffer  n_destinations_extra
         01       Stockholm     124106                 1362                   1587                   225
         14 Västra Götaland      84868                 1013                   1248                   235
         12           Skåne      66906                  819                    934                   115
         05    Östergötland      20441                  280                    428                   148
         03         Uppsala      18124                  249                    631                   382
         13         Halland      17725                  192                    661                   469
         06       Jönköping      16443                  213                    386                   173
         04    Södermanland      14093                  181                    572                   391
         18          Örebro   

## 3. Update r5r Configuration

Add per-county configurations to `r_scripts/r5r_config.json`.

In [8]:
# Load existing config
config_path = Path('r_scripts/r5r_config.json')
with open(config_path) as f:
    config = json.load(f)

# Add per-county configurations
for county_code in SOUTH_COUNTY_CODES:
    city_key = f'sweden_c_{county_code}'
    config['cities'][city_key] = {
        'transport_network_dir': f'dbs/gtfs/sweden_south/c_{county_code}',
        'poi_file': f'dbs/routing/sweden_c_{county_code}/poi_locations.csv',
        'output_dir': f'dbs/routing/sweden_c_{county_code}',
        'destinations_file': f'dbs/routing/sweden_c_{county_code}/deso_centroids.csv',
        'timezone': 'Europe/Stockholm'
    }

# Remove old sweden_south config if exists
if 'sweden_south' in config['cities']:
    del config['cities']['sweden_south']

# Save updated config
with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Updated: {config_path}")
print(f"\nAdded {len(SOUTH_COUNTY_CODES)} county configurations:")
for county_code in SOUTH_COUNTY_CODES:
    print(f"  sweden_c_{county_code} ({COUNTY_NAMES[county_code]})")

Updated: r_scripts/r5r_config.json

Added 15 county configurations:
  sweden_c_01 (Stockholm)
  sweden_c_03 (Uppsala)
  sweden_c_04 (Södermanland)
  sweden_c_05 (Östergötland)
  sweden_c_06 (Jönköping)
  sweden_c_07 (Kronoberg)
  sweden_c_08 (Kalmar)
  sweden_c_09 (Gotland)
  sweden_c_10 (Blekinge)
  sweden_c_12 (Skåne)
  sweden_c_13 (Halland)
  sweden_c_14 (Västra Götaland)
  sweden_c_17 (Värmland)
  sweden_c_18 (Örebro)
  sweden_c_19 (Västmanland)


## 4. Verify GTFS Directory Structure

Check that each county has GTFS and OSM data.

In [9]:
# Check GTFS directories
print("GTFS Directory Status:")
print("=" * 60)

for county_code in SOUTH_COUNTY_CODES:
    gtfs_dir = GTFS_BASE_DIR / f'c_{county_code}'
    
    if not gtfs_dir.exists():
        print(f"c_{county_code} ({COUNTY_NAMES[county_code]}): MISSING")
        continue
    
    files = list(gtfs_dir.glob('*'))
    gtfs_files = [f for f in files if f.suffix == '.zip']
    osm_files = [f for f in files if f.suffix == '.pbf']
    network_files = [f for f in files if f.name == 'network.dat']
    
    status = []
    if gtfs_files:
        status.append(f"{len(gtfs_files)} GTFS")
    if osm_files:
        status.append(f"{len(osm_files)} OSM")
    if network_files:
        status.append("network built")
    
    print(f"c_{county_code} ({COUNTY_NAMES[county_code]}): {', '.join(status) if status else 'empty'}")

GTFS Directory Status:
c_01 (Stockholm): 1 GTFS, 1 OSM, network built
c_03 (Uppsala): 1 GTFS, 1 OSM, network built
c_04 (Södermanland): 1 GTFS, 1 OSM, network built
c_05 (Östergötland): 1 GTFS, 1 OSM, network built
c_06 (Jönköping): 1 GTFS, 1 OSM, network built
c_07 (Kronoberg): 1 GTFS, 1 OSM, network built
c_08 (Kalmar): 1 GTFS, 1 OSM, network built
c_09 (Gotland): 1 GTFS, 1 OSM, network built
c_10 (Blekinge): 1 GTFS, 1 OSM, network built
c_12 (Skåne): 1 GTFS, 1 OSM, network built
c_13 (Halland): 1 GTFS, 1 OSM, network built
c_14 (Västra Götaland): 1 GTFS, 1 OSM, network built
c_17 (Värmland): 1 GTFS, 1 OSM, network built
c_18 (Örebro): 1 GTFS, 1 OSM, network built
c_19 (Västmanland): 1 GTFS, 1 OSM, network built


## 5. Summary

### Output Structure

```
dbs/routing/
├── sweden_c_01/
│   ├── poi_locations.csv      # Origins (Stockholm POIs)
│   └── deso_centroids.csv     # Destinations (Stockholm + 20km buffer)
├── sweden_c_03/
│   ├── poi_locations.csv      # Origins (Uppsala POIs)
│   └── deso_centroids.csv     # Destinations (Uppsala + 20km buffer)
└── ...
```

### GTFS Structure

```
dbs/gtfs/sweden_south/
├── c_01/                      # Stockholm
│   ├── sweden-01.osm.pbf
│   ├── sweden_fixed.zip       # GTFS
│   └── network.dat            # Built network
├── c_03/                      # Uppsala
└── ...
```

### Running Transit Catchment

```bash
# Run for a single county
Rscript r_scripts/compute_transit_catchment.R --city sweden_c_01 --max_time 45

# Run for all counties (bash loop)
for c in 01 03 04 05 06 07 08 09 10 12 13 14 17 18 19; do
    echo "Processing county $c..."
    Rscript r_scripts/compute_transit_catchment.R --city sweden_c_$c --max_time 45
done
```

In [10]:
# Final summary
print("=" * 70)
print("SWEDEN PER-COUNTY TRANSIT CATCHMENT DATA PREPARATION COMPLETE")
print("=" * 70)

print(f"\nCounties processed: {len(all_stats)}")
print(f"Total POIs (origins): {stats_df['n_origins'].sum():,}")
print(f"Buffer distance: {BUFFER_METERS / 1000:.0f} km")

print("\nPer-county summary:")
for _, row in stats_df.iterrows():
    print(f"  {row['county_code']} - {row['county_name']}: "
          f"{row['n_origins']:,} POIs, "
          f"{row['n_destinations_buffer']:,} DeSO ({row['n_destinations_extra']:+,} from buffer)")

print(f"\nConfiguration updated in: r_scripts/r5r_config.json")
print("\nReady to run transit routing per county!")

SWEDEN PER-COUNTY TRANSIT CATCHMENT DATA PREPARATION COMPLETE

Counties processed: 15
Total POIs (origins): 435,278
Buffer distance: 20 km

Per-county summary:
  01 - Stockholm: 124,106 POIs, 1,587 DeSO (+225 from buffer)
  14 - Västra Götaland: 84,868 POIs, 1,248 DeSO (+235 from buffer)
  12 - Skåne: 66,906 POIs, 934 DeSO (+115 from buffer)
  05 - Östergötland: 20,441 POIs, 428 DeSO (+148 from buffer)
  03 - Uppsala: 18,124 POIs, 631 DeSO (+382 from buffer)
  13 - Halland: 17,725 POIs, 661 DeSO (+469 from buffer)
  06 - Jönköping: 16,443 POIs, 386 DeSO (+173 from buffer)
  04 - Södermanland: 14,093 POIs, 572 DeSO (+391 from buffer)
  18 - Örebro: 13,469 POIs, 283 DeSO (+99 from buffer)
  17 - Värmland: 13,249 POIs, 247 DeSO (+66 from buffer)
  08 - Kalmar: 13,171 POIs, 255 DeSO (+97 from buffer)
  19 - Västmanland: 11,919 POIs, 300 DeSO (+141 from buffer)
  07 - Kronoberg: 9,323 POIs, 246 DeSO (+131 from buffer)
  10 - Blekinge: 7,349 POIs, 160 DeSO (+67 from buffer)
  09 - Gotland: 4

In [13]:
gtfs_dir = Path("dbs/gtfs/sweden_south/c_01")

def read_csv_from_zip(zf, name):
    with zf.open(name) as f:
        return pd.read_csv(f, dtype=str)

rows = []

for fp in gtfs_dir.glob("*.zip"):
    try:
        with zipfile.ZipFile(fp) as zf:
            has_cal = "calendar.txt" in zf.namelist()
            has_cal_dates = "calendar_dates.txt" in zf.namelist()

            start_min = end_max = None
            cal_dates_min = cal_dates_max = None

            if has_cal:
                cal = read_csv_from_zip(zf, "calendar.txt")
                if "start_date" in cal.columns:
                    start_min = cal["start_date"].min()
                if "end_date" in cal.columns:
                    end_max = cal["end_date"].max()

            if has_cal_dates:
                cd = read_csv_from_zip(zf, "calendar_dates.txt")
                if "date" in cd.columns and len(cd) > 0:
                    cal_dates_min = cd["date"].min()
                    cal_dates_max = cd["date"].max()

            rows.append({
                "feed": fp.name,
                "has_calendar": has_cal,
                "has_calendar_dates": has_cal_dates,
                "calendar_start_min": start_min,
                "calendar_end_max": end_max,
                "calendar_dates_min": cal_dates_min,
                "calendar_dates_max": cal_dates_max,
            })
    except Exception as e:
        rows.append({"feed": fp.name, "error": str(e)})

df = pd.DataFrame(rows)
print(df)

               feed  has_calendar  has_calendar_dates calendar_start_min  \
0  sweden_fixed.zip          True                True           20221211   

  calendar_end_max calendar_dates_min calendar_dates_max  
0         20241214           20221211           20241214  
